In [1]:
import pandas as pd

# Load the data
data_path = r'C:\Users\sunka\Documents\victoria-road-crash-analysis\data\raw'

accident = pd.read_csv(data_path + r'\accident.csv')
person   = pd.read_csv(data_path + r'\person.csv')
vehicle  = pd.read_csv(data_path + r'\vehicle.csv')
node     = pd.read_csv(data_path + r'\node.csv')

print("All 4 tables loaded successfully!")
print(f"Accident table: {accident.shape}")

All 4 tables loaded successfully!
Accident table: (200352, 23)


In [2]:
# PROBLEM: ACCIDENT_DATE is stored as text, not a real date
# SOLUTION: Convert it to a proper date format

print("Before fixing:")
print(f"ACCIDENT_DATE type: {accident['ACCIDENT_DATE'].dtype}")
print(f"Example value: {accident['ACCIDENT_DATE'][0]}")

# Fix it - convert text to proper date
accident['ACCIDENT_DATE'] = pd.to_datetime(accident['ACCIDENT_DATE'])

print("\nAfter fixing:")
print(f"ACCIDENT_DATE type: {accident['ACCIDENT_DATE'].dtype}")
print(f"Example value: {accident['ACCIDENT_DATE'][0]}")

Before fixing:
ACCIDENT_DATE type: str
Example value: 2012-01-31

After fixing:
ACCIDENT_DATE type: datetime64[us]
Example value: 2012-01-31 00:00:00


In [3]:
# Now that it's a real date, we can extract useful information
accident['YEAR']        = accident['ACCIDENT_DATE'].dt.year
accident['MONTH']       = accident['ACCIDENT_DATE'].dt.month
accident['MONTH_NAME']  = accident['ACCIDENT_DATE'].dt.strftime('%B')
accident['DAY_OF_WEEK'] = accident['ACCIDENT_DATE'].dt.day_name()
accident['HOUR']        = pd.to_datetime(accident['ACCIDENT_TIME'], 
                          format='%H:%M:%S').dt.hour

print("New columns added successfully!")
print(accident[['ACCIDENT_DATE','YEAR','MONTH','MONTH_NAME',
                'DAY_OF_WEEK','HOUR']].head(10))

New columns added successfully!
  ACCIDENT_DATE  YEAR  MONTH MONTH_NAME DAY_OF_WEEK  HOUR
0    2012-01-31  2012      1    January     Tuesday    10
1    2012-02-02  2012      2   February    Thursday     7
2    2012-02-10  2012      2   February      Friday    14
3    2012-02-17  2012      2   February      Friday     8
4    2012-02-24  2012      2   February      Friday    15
5    2012-03-14  2012      3      March   Wednesday    12
6    2012-04-01  2012      4      April      Sunday     8
7    2012-04-03  2012      4      April     Tuesday     1
8    2012-04-07  2012      4      April    Saturday     1
9    2012-02-16  2012      2   February    Thursday     9


In [4]:
# PROBLEM: SEVERITY is just numbers 1,2,3,4 - meaningless without context
# SOLUTION: Map numbers to their real meaning using the data dictionary

severity_map = {
    1: 'Fatal',
    2: 'Serious Injury',
    3: 'Other Injury',
    4: 'Non Injury'
}

accident['SEVERITY_DESC'] = accident['SEVERITY'].map(severity_map)

print("Severity decoded!")
print(accident[['SEVERITY','SEVERITY_DESC']].value_counts())

Severity decoded!
SEVERITY  SEVERITY_DESC 
3         Other Injury      124942
2         Serious Injury     72054
1         Fatal               3352
4         Non Injury             4
Name: count, dtype: int64


In [5]:
# Speed zones are numbers - let's see what values exist
print("Speed zones in the data:")
print(accident['SPEED_ZONE'].value_counts().sort_index())

Speed zones in the data:
SPEED_ZONE
30       421
40     12937
50     32987
60     65305
70     12540
75        20
80     30377
90       505
100    28218
110     2087
777      394
888     1342
999    13219
Name: count, dtype: int64


In [6]:
print("Speed zones in the data:")
print(accident['SPEED_ZONE'].value_counts().sort_index())

Speed zones in the data:
SPEED_ZONE
30       421
40     12937
50     32987
60     65305
70     12540
75        20
80     30377
90       505
100    28218
110     2087
777      394
888     1342
999    13219
Name: count, dtype: int64


In [7]:
# Remove invalid speed zone codes
invalid_zones = [777, 888, 999]
accident_clean = accident[~accident['SPEED_ZONE'].isin(invalid_zones)]

print(f"Rows before cleaning: {len(accident):,}")
print(f"Rows after removing invalid zones: {len(accident_clean):,}")
print(f"Rows removed: {len(accident) - len(accident_clean):,}")

# Verify they're gone
print("\nSpeed zones after cleaning:")
print(accident_clean['SPEED_ZONE'].value_counts().sort_index())

Rows before cleaning: 200,352
Rows after removing invalid zones: 185,397
Rows removed: 14,955

Speed zones after cleaning:
SPEED_ZONE
30       421
40     12937
50     32987
60     65305
70     12540
75        20
80     30377
90       505
100    28218
110     2087
Name: count, dtype: int64


In [8]:
# Check person table columns
print("Person table columns:")
print(person.columns.tolist())

print("\nFirst 3 rows:")
print(person.head(3))

print("\nAge group values:")
print(person['AGE_GROUP'].value_counts())

Person table columns:
['ACCIDENT_NO', 'PERSON_ID', 'VEHICLE_ID', 'SEX', 'AGE_GROUP', 'INJ_LEVEL', 'INJ_LEVEL_DESC', 'SEATING_POSITION', 'HELMET_BELT_WORN', 'ROAD_USER_TYPE', 'ROAD_USER_TYPE_DESC', 'LICENCE_STATE', 'TAKEN_HOSPITAL', 'EJECTED_CODE']

First 3 rows:
    ACCIDENT_NO PERSON_ID VEHICLE_ID SEX AGE_GROUP  INJ_LEVEL INJ_LEVEL_DESC  \
0  T20120002501         C          C   M     50-59          4    Not injured   
1  T20120002923         C          C   U   Unknown          4    Not injured   
2  T20120004530         A          A   M     40-49          3   Other injury   

  SEATING_POSITION  HELMET_BELT_WORN  ROAD_USER_TYPE ROAD_USER_TYPE_DESC  \
0                D               1.0               2             Drivers   
1                D               9.0               2             Drivers   
2                D               1.0               2             Drivers   

  LICENCE_STATE TAKEN_HOSPITAL  EJECTED_CODE  
0             V            NaN           0.0  
1             Z  

In [9]:
# Check what SEX values exist
print("SEX values:")
print(person['SEX'].value_counts())

print("\nINJ_LEVEL values:")
print(person['INJ_LEVEL_DESC'].value_counts())

# Remove unknown sex and unknown age
person_clean = person[
    (person['SEX'].isin(['M','F'])) & 
    (person['AGE_GROUP'] != 'Unknown')
]

print(f"\nPerson rows before: {len(person):,}")
print(f"Person rows after: {len(person_clean):,}")
print(f"Removed: {len(person) - len(person_clean):,}")

SEX values:
SEX
M    263362
F    187659
U     16675
Name: count, dtype: int64

INJ_LEVEL values:
INJ_LEVEL_DESC
Not injured       207300
Other injury      173134
Serious injury     83690
Fatality            3606
Name: count, dtype: int64

Person rows before: 467,730
Person rows after: 448,386
Removed: 19,344


In [10]:
# Get only FATAL cases from person table
fatal_persons = person_clean[person_clean['INJ_LEVEL_DESC'] == 'Fatality']

print(f"Total fatalities: {len(fatal_persons):,}")

print("\nFatalities by SEX:")
print(fatal_persons['SEX'].value_counts())

print("\nFatalities by AGE GROUP:")
print(fatal_persons['AGE_GROUP'].value_counts().sort_index())

print("\nFatalities by ROAD USER TYPE:")
print(fatal_persons['ROAD_USER_TYPE_DESC'].value_counts())

Total fatalities: 3,595

Fatalities by SEX:
SEX
M    2593
F    1002
Name: count, dtype: int64

Fatalities by AGE GROUP:
AGE_GROUP
0-4       30
13-15     40
16-17     78
18-21    335
22-25    271
26-29    277
30-39    534
40-49    445
5-12      43
50-59    445
60-64    213
65-69    196
70+      688
Name: count, dtype: int64

Fatalities by ROAD USER TYPE:
ROAD_USER_TYPE_DESC
Drivers               1697
Motorcyclists          608
Passengers             579
Pedestrians            545
Bicyclists             135
Not Known               16
Pillion Passengers      11
E-scooter Rider          4
Name: count, dtype: int64


In [11]:
# Join accident and person tables together
# This lets us connect WHO died with WHEN and WHERE
merged = pd.merge(
    fatal_persons,
    accident_clean[['ACCIDENT_NO','YEAR','MONTH','MONTH_NAME',
                    'DAY_OF_WEEK','HOUR','SEVERITY_DESC','SPEED_ZONE']],
    on='ACCIDENT_NO',
    how='left'
)

print(f"Merged table rows: {len(merged):,}")
print(f"Columns: {merged.columns.tolist()}")

Merged table rows: 3,595
Columns: ['ACCIDENT_NO', 'PERSON_ID', 'VEHICLE_ID', 'SEX', 'AGE_GROUP', 'INJ_LEVEL', 'INJ_LEVEL_DESC', 'SEATING_POSITION', 'HELMET_BELT_WORN', 'ROAD_USER_TYPE', 'ROAD_USER_TYPE_DESC', 'LICENCE_STATE', 'TAKEN_HOSPITAL', 'EJECTED_CODE', 'YEAR', 'MONTH', 'MONTH_NAME', 'DAY_OF_WEEK', 'HOUR', 'SEVERITY_DESC', 'SPEED_ZONE']


In [12]:
# QUESTION 1: When are fatal crashes most likely?

print("=== FATALITIES BY HOUR OF DAY ===")
print(merged['HOUR'].value_counts().sort_index())

print("\n=== FATALITIES BY DAY OF WEEK ===")
days_order = ['Monday','Tuesday','Wednesday',
              'Thursday','Friday','Saturday','Sunday']
day_counts = merged['DAY_OF_WEEK'].value_counts()
for day in days_order:
    print(f"{day}: {day_counts.get(day, 0)}")

print("\n=== FATALITIES BY YEAR (trend) ===")
print(merged['YEAR'].value_counts().sort_index())

=== FATALITIES BY HOUR OF DAY ===
HOUR
0.0     104
1.0      90
2.0      77
3.0      68
4.0      53
5.0     103
6.0     156
7.0     133
8.0     142
9.0     149
10.0    150
11.0    189
12.0    192
13.0    199
14.0    220
15.0    255
16.0    251
17.0    191
18.0    181
19.0    145
20.0    140
21.0    125
22.0    111
23.0    109
Name: count, dtype: int64

=== FATALITIES BY DAY OF WEEK ===
Monday: 464
Tuesday: 427
Wednesday: 466
Thursday: 518
Friday: 554
Saturday: 565
Sunday: 539

=== FATALITIES BY YEAR (trend) ===
YEAR
2012.0    280
2013.0    239
2014.0    248
2015.0    249
2016.0    287
2017.0    253
2018.0    208
2019.0    259
2020.0    206
2021.0    225
2022.0    237
2023.0    288
2024.0    275
2025.0    279
Name: count, dtype: int64


In [13]:
# QUESTION 3: Which speed zones are most deadly?
print("=== FATALITIES BY SPEED ZONE ===")
print(merged['SPEED_ZONE'].value_counts().sort_index())

# Calculate fatality RATE - fatalities per total crashes
print("\n=== FATALITY RATE BY SPEED ZONE ===")
total_by_zone = accident_clean['SPEED_ZONE'].value_counts()
fatal_by_zone = merged['SPEED_ZONE'].value_counts()

for zone in sorted(total_by_zone.index):
    total = total_by_zone.get(zone, 0)
    fatal = fatal_by_zone.get(zone, 0)
    rate = (fatal / total * 100)
    print(f"{zone} km/h: {fatal} deaths from {total:,} crashes = {rate:.1f}% fatal rate")

=== FATALITIES BY SPEED ZONE ===
SPEED_ZONE
30.0        2
40.0       74
50.0      307
60.0      695
70.0      198
75.0        1
80.0      616
90.0       22
100.0    1497
110.0     121
Name: count, dtype: int64

=== FATALITY RATE BY SPEED ZONE ===
30 km/h: 2 deaths from 421 crashes = 0.5% fatal rate
40 km/h: 74 deaths from 12,937 crashes = 0.6% fatal rate
50 km/h: 307 deaths from 32,987 crashes = 0.9% fatal rate
60 km/h: 695 deaths from 65,305 crashes = 1.1% fatal rate
70 km/h: 198 deaths from 12,540 crashes = 1.6% fatal rate
75 km/h: 1 deaths from 20 crashes = 5.0% fatal rate
80 km/h: 616 deaths from 30,377 crashes = 2.0% fatal rate
90 km/h: 22 deaths from 505 crashes = 4.4% fatal rate
100 km/h: 1497 deaths from 28,218 crashes = 5.3% fatal rate
110 km/h: 121 deaths from 2,087 crashes = 5.8% fatal rate


In [14]:
# Export the clean data for Tableau dashboard
output_path = r'C:\Users\sunka\Documents\victoria-road-crash-analysis\data\processed'

# Export 1 - cleaned accident table
accident_clean.to_csv(output_path + r'\accident_clean.csv', index=False)

# Export 2 - fatal persons merged table
merged.to_csv(output_path + r'\fatal_analysis.csv', index=False)

# Export 3 - fatality rate by speed zone
speed_summary = []
for zone in sorted(total_by_zone.index):
    total = total_by_zone.get(zone, 0)
    fatal = fatal_by_zone.get(zone, 0)
    rate = round(fatal / total * 100, 1)
    speed_summary.append({'SPEED_ZONE': zone, 
                          'TOTAL_CRASHES': total,
                          'FATALITIES': fatal, 
                          'FATAL_RATE_PCT': rate})

speed_df = pd.DataFrame(speed_summary)
speed_df.to_csv(output_path + r'\speed_zone_summary.csv', index=False)

print("✅ All files exported successfully!")
print(f"\nFiles saved to: {output_path}")
print("- accident_clean.csv")
print("- fatal_analysis.csv") 
print("- speed_zone_summary.csv")

✅ All files exported successfully!

Files saved to: C:\Users\sunka\Documents\victoria-road-crash-analysis\data\processed
- accident_clean.csv
- fatal_analysis.csv
- speed_zone_summary.csv
